# Reinforcement Learning — Math Foundations, Worked in Code

This notebook is the computational companion to the *Actor–Critic & Beyond* slide deck.
Every concept from the deck gets the same treatment: **the formula, then real numbers
running through it in Python.** Richard Feynman's test for understanding something is
being able to compute it from scratch — that's what every code cell here does.

A live LLM (hosted on Groq) is wired in as an on-demand tutor: call `ask_groq("...")`
anywhere to get a plain-language explanation of whatever you just computed.

**Structure** (mirrors the slide deck's sections):
1. Foundations — MDPs, Bellman equation, the policy-gradient derivation, TD-error
2. Core Concepts — exploration, on/off-policy, Q-learning, DQN, bias-variance, GAE,
   importance sampling, eligibility traces, function approximation, action spaces
3. Actor-Critic Family — A2C, PPO, TRPO, DDPG, TD3, SAC
4. Model-Based RL — Dyna-Q, MCTS, AlphaZero, MuZero, Dreamer
5. Multi-Agent RL — VDN, QMIX, MADDPG
6. Advanced Topics — Distributional RL, Curiosity, RLHF
7. Wrap-up — ask the tutor anything

## 0. Setup

Groq's API key goes into an environment variable — never hardcode a *real* key in a
notebook you intend to share or commit to git; use `getpass()` instead so it's typed in
at runtime and never saved to disk. The dummy key below is only for this demo.

In [ ]:
import os
from getpass import getpass

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = "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"

# If you'd rather type your own key at runtime instead of hardcoding one, use:
# if "GROQ_API_KEY" not in os.environ:
#     os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

In [ ]:
%pip install -q groq numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.facecolor"] = "white"
rng = np.random.default_rng(7)

try:
    from groq import Groq
    _groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])
except Exception as e:
    _groq_client = None
    print(f"(Groq client not created: {e})")

def ask_groq(prompt, model="llama-3.3-70b-versatile"):
    '''Ask the Groq-hosted LLM to explain a concept, Feynman-style.
    Fails gracefully (prints a message instead of raising) if offline or misconfigured,
    so the rest of the notebook keeps running either way.'''
    system = (
        "You are a concise, precise reinforcement-learning tutor. "
        "Explain like Richard Feynman: simple language, concrete intuition, "
        "no unnecessary jargon. Keep answers under 120 words."
    )
    if _groq_client is None:
        return "[Groq client not initialized -- skipping live explanation.]"
    try:
        resp = _groq_client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": prompt},
            ],
            temperature=0.4,
            max_tokens=220,
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        return f"[Groq call failed ({e}) -- check your network/API key. Continuing with the math only.]"

print("Setup complete. Try: print(ask_groq('What is an MDP?'))")

---
# 1. Foundations

## 1.1 The Markov Decision Process

An MDP is the tuple $(S, A, P, R, \gamma)$:

- $S$ — the set of states
- $A$ — the set of actions
- $P(s'|s,a)$ — transition probabilities
- $R(s,a)$ — the reward function
- $\gamma \in [0,1)$ — the discount factor

The **Markov property** is the whole reason this is tractable: $P(s'|s,a)$ depends only
on the current $(s,a)$, never on how you got there.

In [ ]:
# A tiny 3-state MDP, represented explicitly.
states = ["s0", "s1", "s2 (terminal)"]
actions = ["forward"]
gamma = 0.9

# P[s][a] -> {s': probability}
P = {
    "s0": {"forward": {"s1": 1.0}},
    "s1": {"forward": {"s2 (terminal)": 1.0}},
    "s2 (terminal)": {"forward": {"s2 (terminal)": 1.0}},
}
# R[s][a] -> immediate reward
R = {"s0": {"forward": 3.0}, "s1": {"forward": 5.0}, "s2 (terminal)": {"forward": 0.0}}

print("MDP = (S, A, P, R, gamma)")
print("S =", states)
print("A =", actions)
print("gamma =", gamma)
print("R(s0,forward) =", R["s0"]["forward"], "  R(s1,forward) =", R["s1"]["forward"])

## 1.2 The Bellman Equation

$$V(s) = \mathbb{E}\big[\, r + \gamma \, V(s') \,\big]$$

Value is recursive: what a state is worth = the immediate reward, plus the (discounted)
value of wherever you land next. Because the terminal state has *nothing* left to earn,
we can solve this **backward**, one step at a time — no simultaneous equations needed
for a simple chain like this one.

In [ ]:
# Backward induction over the 3-state chain (same numbers as the slide deck).
V2 = 0.0                              # terminal state: nothing left to earn
V1 = R["s1"]["forward"] + gamma * V2  # = 5 + 0.9 * 0
V0 = R["s0"]["forward"] + gamma * V1  # = 3 + 0.9 * 5.0

print(f"V(s2) = {V2:.2f}   (terminal)")
print(f"V(s1) = {R['s1']['forward']} + {gamma} * {V2:.2f}  =  {V1:.2f}")
print(f"V(s0) = {R['s0']['forward']} + {gamma} * {V1:.2f}  =  {V0:.2f}")

For anything with branches or cycles, backward substitution won't work directly — instead
solve the *linear system* $V = R + \gamma P V \;\Rightarrow\; V = (I - \gamma P)^{-1} R$.
Let's verify both methods agree on our simple chain, then show the general linear-algebra
approach on a slightly richer MRP with a self-loop.

In [ ]:
# General solution via linear algebra: V = (I - gamma*P)^-1 R
# States: s0 -> s1 -> s2(terminal, absorbing)
P_matrix = np.array([
    [0.0, 1.0, 0.0],   # from s0: always to s1
    [0.0, 0.0, 1.0],   # from s1: always to s2
    [0.0, 0.0, 1.0],   # s2 absorbing: stays at s2
])
R_vector = np.array([3.0, 5.0, 0.0])

V_linalg = np.linalg.solve(np.eye(3) - gamma * P_matrix, R_vector)
print("V via linear algebra:", V_linalg)
print("V via backward induction:", [V0, V1, V2])
assert np.allclose(V_linalg, [V0, V1, V2]), "mismatch!"
print("Both methods agree. ✓")

# --- A richer MRP with a self-loop, to show why linear algebra is the general tool ---
# s0 -> s1 (r=2);  s1 -> s1 w.p. 0.3 (r=1, "stuck in traffic") or -> s2 w.p. 0.7 (r=4)
P2 = np.array([
    [0.0, 1.0, 0.0],
    [0.0, 0.3, 0.7],
    [0.0, 0.0, 1.0],
])
R2 = np.array([2.0, 0.3 * 1.0 + 0.7 * 4.0, 0.0])  # expected immediate reward per state
V2_general = np.linalg.solve(np.eye(3) - gamma * P2, R2)
print("\nMRP with a self-loop, V =", V2_general)

## 1.3 Why the Policy Gradient Works — the log-derivative trick

We want $\nabla_\theta J(\theta)$ where $J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}[R(\tau)]$.
The problem: *sampling* an action isn't differentiable, so we can't naively backprop
through "sample $a$, then observe $R$."

$$
J(\theta) = \int \pi_\theta(\tau) R(\tau)\, d\tau
\quad\Rightarrow\quad
\nabla_\theta J(\theta) = \int \nabla_\theta \pi_\theta(\tau)\, R(\tau)\, d\tau
$$

The **key trick**: $\nabla_\theta \pi_\theta(\tau) = \pi_\theta(\tau) \nabla_\theta \log \pi_\theta(\tau)$
(just the identity $\nabla_\theta \log f = \nabla_\theta f / f$, rearranged). Substitute it in:

$$
\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\big[\, \nabla_\theta \log \pi_\theta(\tau)\, R(\tau) \,\big]
$$

That's now an expectation — something we CAN estimate, just by acting and averaging.
Let's verify this numerically on the simplest possible case: a 1-state, 2-action bandit.

In [ ]:
# A 1-step "contextual bandit": 2 actions, reward r(0)=1, r(1)=4.
# Policy: pi_theta(a=1) = sigmoid(theta),  pi_theta(a=0) = 1 - sigmoid(theta)
r0, r1 = 1.0, 4.0

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def J_analytical(theta):
    p1 = sigmoid(theta)
    return p1 * r1 + (1 - p1) * r0

def dJ_analytical(theta):
    # dJ/dtheta = (r1 - r0) * p1 * (1 - p1),  since d(sigmoid)/dtheta = p*(1-p)
    p1 = sigmoid(theta)
    return (r1 - r0) * p1 * (1 - p1)

def reinforce_gradient_estimate(theta, n_samples=20000, seed=0):
    '''Monte-Carlo estimate of the SAME gradient using ONLY samples + log-prob --
    exactly the log-derivative-trick expectation, no calculus-through-sampling required.'''
    local_rng = np.random.default_rng(seed)
    p1 = sigmoid(theta)
    actions = (local_rng.random(n_samples) < p1).astype(int)   # sample a ~ pi_theta
    rewards = np.where(actions == 1, r1, r0)
    # grad_theta log pi(a|s):  a=1 -> (1-p1);  a=0 -> -p1
    grad_log_pi = np.where(actions == 1, 1 - p1, -p1)
    return np.mean(grad_log_pi * rewards)   # E[grad_log_pi * R]  <-- the trick, in one line

theta0 = 0.3
print(f"theta = {theta0}")
print(f"Analytical dJ/dtheta        = {dJ_analytical(theta0):.4f}")
print(f"REINFORCE (20,000 samples)  = {reinforce_gradient_estimate(theta0):.4f}")
print("\nThese should match closely -- the sampled estimator IS an unbiased estimate")
print("of the true gradient, purely because of the log-derivative trick.")

In [ ]:
# Visualize convergence of the Monte-Carlo estimate as we throw more samples at it.
sample_sizes = [10, 50, 200, 1000, 5000, 20000, 100000]
estimates = [reinforce_gradient_estimate(theta0, n_samples=n, seed=1) for n in sample_sizes]
true_grad = dJ_analytical(theta0)

plt.figure(figsize=(6, 3.5))
plt.axhline(true_grad, color="#e8a765", linestyle="--", label="analytical gradient")
plt.plot(sample_sizes, estimates, "o-", color="#5fd0c3", label="REINFORCE estimate")
plt.xscale("log")
plt.xlabel("number of samples")
plt.ylabel("gradient estimate")
plt.title("REINFORCE estimate converging to the true gradient")
plt.legend()
plt.tight_layout()
plt.show()

## 1.4 TD-Error — the Critic's one-number judgment

$$\delta = r + \gamma V(s') - V(s)$$

$\delta$ is a cheap, low-variance stand-in for the advantage $A(s,a) = Q(s,a) - V(s)$.
Positive $\delta$ means "better than expected" — nudge that action's probability up.

In [ ]:
r, V_s, V_s_next, gamma = 2.0, 10.0, 9.0, 0.9
delta = r + gamma * V_s_next - V_s

print(f"delta = r + gamma*V(s') - V(s)")
print(f"      = {r} + {gamma}*{V_s_next} - {V_s}")
print(f"      = {delta:.2f}")
print()
if delta > 0:
    print(f"delta > 0: that action did SLIGHTLY better than expected -> small positive nudge.")
else:
    print(f"delta <= 0: that action did worse than expected -> push its probability down.")

In [ ]:
print(ask_groq(
    "In two sentences, explain why the TD-error is a good stand-in for the advantage "
    "function, even though it only looks one step ahead."
))

---
# 2. Core Concepts

## 2.1 Exploration vs. Exploitation — $\varepsilon$-greedy

$$a = \begin{cases} \text{random action} & \text{w.p. } \varepsilon \\ \arg\max_a Q(s,a) & \text{w.p. } 1-\varepsilon \end{cases}$$

In [ ]:
def epsilon_schedule(episode, start=1.0, end=0.05, decay=0.97):
    return max(end, start * (decay ** episode))

def epsilon_greedy(q_values, epsilon, rng=rng):
    if rng.random() < epsilon:
        return rng.integers(len(q_values))          # explore
    return int(np.argmax(q_values))                  # exploit

episodes = np.arange(0, 150)
eps_curve = [epsilon_schedule(e) for e in episodes]

plt.figure(figsize=(6, 3))
plt.plot(episodes, eps_curve, color="#e08a9c")
plt.xlabel("episode"); plt.ylabel("epsilon")
plt.title("epsilon decay over training")
plt.tight_layout(); plt.show()

q_demo = np.array([1.2, 3.7, 0.4])
picks = [epsilon_greedy(q_demo, 0.3) for _ in range(10)]
print("Q-values:", q_demo, " -> greedy action:", np.argmax(q_demo))
print("10 epsilon-greedy picks (eps=0.3):", picks)

## 2.2 On-Policy vs. Off-Policy — the update target is the tell

Both SARSA (on-policy) and Q-learning (off-policy) compute a TD target, but SARSA uses
the action *actually taken* next; Q-learning uses the *best possible* next action,
regardless of what will really happen.

In [ ]:
def sarsa_target(r, gamma, Q, s_next, a_next):
    # a_next is whatever the (epsilon-greedy) policy actually picked
    return r + gamma * Q[s_next, a_next]

def qlearning_target(r, gamma, Q, s_next):
    # off-policy: always bootstrap from the greedy action, no matter what happens next
    return r + gamma * np.max(Q[s_next])

Q_demo = np.array([[0.0, 0.0], [1.2, 3.7], [0.5, 0.9]])   # 3 states x 2 actions
r, gamma, s_next = 1.0, 0.9, 1
a_next_taken = 0   # suppose the exploring policy happened to pick the WORSE action

print("On-policy SARSA target  (uses action actually taken):",
      sarsa_target(r, gamma, Q_demo, s_next, a_next_taken))
print("Off-policy Q-learning target (uses the BEST action):",
      qlearning_target(r, gamma, Q_demo, s_next))
print("\nThey differ exactly because SARSA is honest about what the exploring policy will do,")
print("while Q-learning always assumes the greedy policy from here on.")

## 2.3 Q-Learning — full tabular training loop

$$Q(s,a) \leftarrow Q(s,a) + \alpha\big[\, r + \gamma \max_{a'} Q(s',a') - Q(s,a) \,\big]$$

A 6-state corridor: start at state 0, actions are `left`/`right`, reward +1 only for
reaching state 5 (terminal). Let's watch $Q$ converge.

In [ ]:
N_STATES = 6
GOAL = N_STATES - 1
ACTIONS = [-1, +1]   # left, right

def step(s, a):
    s_next = np.clip(s + a, 0, GOAL)
    r = 1.0 if s_next == GOAL else 0.0
    done = s_next == GOAL
    return s_next, r, done

def train_qlearning(n_episodes=300, alpha=0.5, gamma=0.95, eps_start=1.0, eps_end=0.05, seed=0):
    local_rng = np.random.default_rng(seed)
    Q = np.zeros((N_STATES, len(ACTIONS)))
    v_history = []
    for ep in range(n_episodes):
        eps = epsilon_schedule(ep, eps_start, eps_end)
        s = 0
        for _ in range(50):
            a_idx = epsilon_greedy(Q[s], eps, local_rng)
            s_next, r, done = step(s, ACTIONS[a_idx])
            target = r + gamma * np.max(Q[s_next]) * (not done)
            Q[s, a_idx] += alpha * (target - Q[s, a_idx])
            s = s_next
            if done:
                break
        v_history.append(Q[0].max())
    return Q, v_history

Q_final, v_hist = train_qlearning()
print("Learned Q-table (rows=states, cols=[left,right]):")
print(Q_final)
print("\nGreedy policy from each state:", ["right" if np.argmax(row) == 1 else "left" for row in Q_final[:-1]])

In [ ]:
plt.figure(figsize=(6, 3))
plt.plot(v_hist, color="#6fa8dc")
plt.xlabel("episode"); plt.ylabel("V(start) = max_a Q(0,a)")
plt.title("Q-learning: value estimate converging to the true optimum")
plt.tight_layout(); plt.show()

## 2.4 DQN — replay buffer + target network, computed on a synthetic batch

$$\mathcal{L}(\theta) = \mathbb{E}\Big[\big(r + \gamma \max_{a'} Q_{\theta^-}(s',a') - Q_\theta(s,a)\big)^2\Big]$$

$Q_{\theta^-}$ is a *frozen copy* of the online network, updated only every $C$ steps —
this stops the target from chasing itself every single update.

In [ ]:
# Represent Q as a tiny linear function of hand-picked features, for both online (theta)
# and target (theta_minus) networks, and compute one DQN-style loss + gradient step.
n_features, n_actions = 4, 2
theta = rng.normal(size=(n_features, n_actions)) * 0.1
theta_minus = theta.copy()   # target net starts as an exact copy

def Q_forward(theta_w, phi):        # phi: (batch, n_features)
    return phi @ theta_w            # -> (batch, n_actions)

batch_size = 8
phi_s      = rng.normal(size=(batch_size, n_features))
phi_s_next = rng.normal(size=(batch_size, n_features))
actions    = rng.integers(0, n_actions, size=batch_size)
rewards    = rng.normal(loc=1.0, size=batch_size)
dones      = rng.random(batch_size) < 0.1
gamma = 0.95

q_online = Q_forward(theta, phi_s)[np.arange(batch_size), actions]
q_target_next = Q_forward(theta_minus, phi_s_next).max(axis=1)
target = rewards + gamma * q_target_next * (~dones)

loss_before = np.mean((target - q_online) ** 2)
print(f"Batch MSE loss BEFORE update: {loss_before:.4f}")

# One manual gradient step on theta (target network theta_minus stays frozen).
lr = 0.05
grad = np.zeros_like(theta)
for i in range(batch_size):
    err = q_online[i] - target[i]
    grad[:, actions[i]] += 2 * err * phi_s[i] / batch_size
theta -= lr * grad

q_online_after = Q_forward(theta, phi_s)[np.arange(batch_size), actions]
loss_after = np.mean((target - q_online_after) ** 2)
print(f"Batch MSE loss AFTER 1 update: {loss_after:.4f}  (target network untouched)")

## 2.5 Bias vs. Variance — the classic "A-B" batch example (Sutton & Barto)

Eight recorded episodes:

```
A, 0, B, 0        (1 episode: A always goes to B, reward 0)
B, 1  (x6)
B, 0  (x1)
```

Monte Carlo uses only the *actual observed return* after visiting A. Batch TD instead
builds the maximum-likelihood model implied by ALL the data and solves it — effectively
borrowing information from every time B was visited, not just the one time it followed A.

In [ ]:
episodes = [
    [("A", 0.0), ("B", 0.0)],
    [("B", 1.0)], [("B", 1.0)], [("B", 1.0)],
    [("B", 1.0)], [("B", 1.0)], [("B", 1.0)],
    [("B", 0.0)],
]

# --- Monte Carlo estimate of V(A): average the actual return of every episode starting at A ---
returns_from_A = [sum(r for _, r in ep) for ep in episodes if ep[0][0] == "A"]
V_A_mc = np.mean(returns_from_A)

# --- Batch TD / certainty-equivalence estimate ---
# V(B) = average reward observed after every visit to B (there are 8 total: 1 inside the
#        A-episode + 7 standalone)
b_rewards = []
for ep in episodes:
    for state, r in ep:
        if state == "B":
            b_rewards.append(r)
V_B_td = np.mean(b_rewards)
# A always transitions to B with reward 0 -> V(A) = 0 + V(B)  (gamma = 1 for this example)
V_A_td = 0.0 + V_B_td

print(f"V(B) [avg over all 8 visits]     = {V_B_td:.3f}")
print(f"V(A) via Monte Carlo (1 episode) = {V_A_mc:.3f}   <- unbiased, but based on a SINGLE sample")
print(f"V(A) via batch TD (uses all data)= {V_A_td:.3f}   <- biased by the model, but far less noisy")
print()
print("This is the textbook demonstration: MC and TD converge to the SAME answer with infinite")
print("data, but with limited data TD can be more sample-efficient by exploiting the MDP structure.")

## 2.6 Generalized Advantage Estimation (GAE)

$$A_t^{GAE(\lambda)} = \sum_{l=0}^{\infty} (\gamma\lambda)^l\, \delta_{t+l}
\qquad\text{computed backward as}\qquad
A_t = \delta_t + \gamma\lambda\, A_{t+1}$$

$\lambda=0$ recovers pure 1-step TD; $\lambda=1$ recovers Monte Carlo.

In [ ]:
deltas = np.array([0.5, -0.2, 0.8, 0.1, -0.3])   # a short trajectory of TD-errors
gamma = 0.99

def gae(deltas, gamma, lam):
    T = len(deltas)
    A = np.zeros(T)
    running = 0.0
    for t in reversed(range(T)):
        running = deltas[t] + gamma * lam * running
        A[t] = running
    return A

print(f"{'lambda':>8} | {'A_0':>8} | full advantage sequence")
for lam in [0.0, 0.5, 0.9, 0.95, 1.0]:
    A = gae(deltas, gamma, lam)
    print(f"{lam:8.2f} | {A[0]:8.3f} | {np.round(A, 3)}")

In [ ]:
lams = np.linspace(0, 1, 40)
A0_vals = [gae(deltas, gamma, l)[0] for l in lams]
plt.figure(figsize=(6, 3))
plt.plot(lams, A0_vals, color="#a98fd2")
plt.xlabel("lambda"); plt.ylabel("A_0 (advantage at t=0)")
plt.title("GAE: sliding from pure TD (lambda=0) to pure MC (lambda=1)")
plt.tight_layout(); plt.show()

## 2.7 Importance Sampling

$$\mathbb{E}_{x \sim p}[f(x)] = \mathbb{E}_{x \sim q}\Big[\frac{p(x)}{q(x)} f(x)\Big]$$

Estimate an expectation under $p$ using only samples from a *different* distribution $q$,
by reweighting each sample. This is the machinery behind PPO's ratio $r(\theta)$ and any
off-policy correction.

In [ ]:
def gaussian_pdf(x, mu, sigma):
    return np.exp(-0.5 * ((x - mu) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))

mu_p, sigma_p = 2.0, 1.0     # target distribution p (what we WANT the expectation under)
mu_q, sigma_q = 0.0, 1.5     # behavior distribution q (what we can actually SAMPLE from)
f = lambda x: x ** 2

n = 200_000
x_p = rng.normal(mu_p, sigma_p, n)          # ground truth: sample directly from p
x_q = rng.normal(mu_q, sigma_q, n)          # samples we actually have, from q
weights = gaussian_pdf(x_q, mu_p, sigma_p) / gaussian_pdf(x_q, mu_q, sigma_q)

true_estimate = np.mean(f(x_p))
naive_estimate = np.mean(f(x_q))                       # WRONG: ignores the mismatch
is_estimate = np.mean(weights * f(x_q))                 # correct: importance-weighted

print(f"True  E_p[f(x)]  (direct sampling from p) = {true_estimate:.3f}")
print(f"Naive E_q[f(x)]  (ignoring the mismatch)   = {naive_estimate:.3f}   <- wrong distribution!")
print(f"IS estimate using samples from q, reweighted = {is_estimate:.3f}   <- corrected")
print(f"\nVariance of weights: {weights.var():.3f}  (this is the classic IS cost --")
print("far from q, a few huge weights can dominate the estimate and blow up variance)")

## 2.8 Eligibility Traces — TD($\lambda$), backward view

$$e_t(s) = \gamma\lambda\, e_{t-1}(s) + \mathbb{1}[s = s_t] \qquad V(s) \leftarrow V(s) + \alpha\,\delta_t\, e_t(s)$$

Every recently-visited state keeps a fading "trace." When a reward finally lands, it
updates every state in the trail at once, weighted by how recently it was visited.

In [ ]:
gamma, lam = 0.9, 0.833   # gamma*lambda = 0.75, matching the slide's decay constant
trajectory = ["s0", "s1", "s2", "s3", "s4", "s5"]   # visited in order
final_reward = 10.0
alpha = 0.5

V = {s: 0.0 for s in trajectory}
e = {s: 0.0 for s in trajectory}

for t, s in enumerate(trajectory):
    # decay all traces, then bump the current state's trace by 1
    for s2 in e:
        e[s2] *= gamma * lam
    e[s] += 1.0
    # only the LAST transition has a nonzero reward in this simple example
    r = final_reward if t == len(trajectory) - 1 else 0.0
    s_next_value = 0.0   # terminal
    delta = r + gamma * s_next_value - V[s]
    for s2 in trajectory:
        V[s2] += alpha * delta * e[s2]
    print(f"after visiting {s}: trace = {[round(e[s2], 3) for s2 in trajectory]}")

print("\nFinal V:", {s: round(v, 3) for s, v in V.items()})
print("Notice how V decays for states visited LONGER before the reward -- exactly (gamma*lambda)^k.")

## 2.9 Function Approximation — generalizing beyond a lookup table

A table has one independent number per state; a linear (or neural) function
$V_\theta(s) = \theta \cdot \phi(s)$ shares information across *similar* states via
features $\phi$, so it can even estimate states it has never visited.

In [ ]:
# 5 known states with hand-labeled "true" values, described by a feature (position on a line).
positions = np.array([0.0, 1.0, 2.0, 3.0, 4.0])
true_values = np.array([0.0, 1.5, 4.0, 7.5, 12.0])   # roughly quadratic in position

def features(x):
    return np.stack([np.ones_like(x), x, x ** 2], axis=-1)   # phi(s) = [1, x, x^2]

Phi = features(positions)
theta = np.zeros(3)
lr = 0.02
for _ in range(2000):
    pred = Phi @ theta
    grad = Phi.T @ (pred - true_values) / len(positions)
    theta -= lr * grad

print("Fitted theta:", theta)
print("Predicted vs true on TRAINING states:", np.round(Phi @ theta, 2), "vs", true_values)

unseen_x = np.array([1.5, 3.5])   # never seen during fitting
unseen_pred = features(unseen_x) @ theta
print(f"\nGeneralizes to UNSEEN states x={unseen_x}: predicted V = {np.round(unseen_pred, 2)}")
print("A table would have NO entry at all for these -- this is why deep RL needs function approximation.")

## 2.10 Discrete vs. Continuous Action Distributions

Discrete actions use a softmax over logits; continuous actions typically use a Gaussian.
Both need a `log_prob(a)` for the policy gradient — that's the $\nabla_\theta \log \pi_\theta(a|s)$
term from section 1.3.

In [ ]:
def softmax(logits):
    z = logits - logits.max()
    e = np.exp(z)
    return e / e.sum()

logits = np.array([1.0, 3.0, 0.5])
probs = softmax(logits)
print("Discrete: logits =", logits, " -> probs =", np.round(probs, 3))
print("log pi(a=1|s) =", np.log(probs[1]).round(4))

def gaussian_log_prob(a, mu, sigma):
    return -0.5 * np.log(2 * np.pi * sigma ** 2) - 0.5 * ((a - mu) / sigma) ** 2

mu, sigma = 0.4, 0.2   # policy outputs a MEAN and STD for the action
a_sample = rng.normal(mu, sigma)
print(f"\nContinuous: sampled action a = {a_sample:.3f} from N({mu}, {sigma}^2)")
print(f"log pi(a|s) = {gaussian_log_prob(a_sample, mu, sigma):.4f}")

---
# 3. Actor-Critic Family

## 3.1 A2C — synchronized advantage across parallel workers

$$\theta \leftarrow \theta + \alpha \cdot \frac{1}{N}\sum_{i=1}^N \nabla_\theta \log \pi_\theta(a_i|s_i)\, A_i$$

In [ ]:
N_workers = 4
log_probs = np.array([-1.2, -0.7, -2.1, -0.9])   # log pi(a_i|s_i) for each worker
advantages = np.array([0.6, -0.3, 1.1, 0.2])
values_pred = np.array([2.1, 3.4, 1.0, 2.8])
values_target = values_pred + advantages   # since A = target - V_pred

policy_loss = -np.mean(log_probs * advantages)     # average across workers = "synchronized" step
value_loss = np.mean((values_target - values_pred) ** 2)
entropy_bonus = 0.15   # encourages exploration; subtracted from the total loss

c1, c2 = 0.5, 0.01
total_loss = policy_loss + c1 * value_loss - c2 * entropy_bonus

print("Per-worker: log_prob * advantage =", np.round(log_probs * advantages, 3))
print(f"Policy loss (averaged over {N_workers} workers): {policy_loss:.4f}")
print(f"Value loss (critic MSE):                          {value_loss:.4f}")
print(f"Total loss = policy + c1*value - c2*entropy:       {total_loss:.4f}")

## 3.2 PPO — the clipped surrogate objective

$$r(\theta) = \frac{\pi_\theta(a|s)}{\pi_{old}(a|s)}
\qquad
L^{CLIP} = \mathbb{E}\big[\min(r(\theta)\hat{A},\; \text{clip}(r(\theta), 1-\varepsilon, 1+\varepsilon)\hat{A})\big]$$

In [ ]:
eps = 0.2
r = np.linspace(0.2, 2.0, 400)

def ppo_clip_objective(r, A, eps=0.2):
    clipped = np.clip(r, 1 - eps, 1 + eps)
    return np.minimum(r * A, clipped * A)

L_pos = ppo_clip_objective(r, A=1.0, eps=eps)    # good action (A > 0): don't overshoot
L_neg = ppo_clip_objective(r, A=-1.0, eps=eps)   # bad action (A < 0): don't overcorrect

plt.figure(figsize=(6.5, 3.5))
plt.plot(r, L_pos, color="#5fd0c3", label="A = +1 (good action)")
plt.plot(r, L_neg, color="#e08a9c", label="A = -1 (bad action)")
plt.axvline(1 - eps, color="#8b8a9a", linestyle=":"); plt.axvline(1 + eps, color="#8b8a9a", linestyle=":")
plt.xlabel("ratio r(theta)"); plt.ylabel("L_CLIP")
plt.title("PPO objective flattens once r(theta) leaves [1-eps, 1+eps]")
plt.legend(); plt.tight_layout(); plt.show()

for test_r in [0.7, 1.0, 1.15, 1.5]:
    print(f"r={test_r:.2f}: L_CLIP(A=+1) = {ppo_clip_objective(test_r, 1.0):.3f}   "
          f"L_CLIP(A=-1) = {ppo_clip_objective(test_r, -1.0):.3f}")

## 3.3 TRPO — trust region via KL divergence and the natural gradient

$$D_{KL}(p \,\|\, q) = \sum_i p_i \log\frac{p_i}{q_i}
\qquad
\Delta\theta = \sqrt{\frac{2\delta}{g^\top F^{-1} g}}\; F^{-1} g$$

$F$ (the Fisher information matrix) is the local curvature of the KL divergence — near
$\theta_{old}$, KL behaves like a quadratic bowl, and $F$ is exactly that bowl's shape.

In [ ]:
def kl_divergence(p, q):
    return np.sum(p * np.log(p / q))

p_old = np.array([0.5, 0.3, 0.2])
p_new = np.array([0.55, 0.25, 0.20])
print(f"KL(p_old || p_new) = {kl_divergence(p_old, p_new):.5f}  (small update -> small KL)")

p_far = np.array([0.9, 0.05, 0.05])
print(f"KL(p_old || p_far) = {kl_divergence(p_old, p_far):.5f}  (big update -> big KL)")

In [ ]:
# Fisher information for a 1-parameter categorical policy, via finite differences of KL.
# Policy: pi_theta = softmax([theta, 0, 0])  (only the first logit moves)
def policy_theta(theta):
    logits = np.array([theta, 0.0, 0.0])
    return softmax(logits)

theta_old = 0.3
eps_fd = 1e-4

def kl_from_theta(theta):
    return kl_divergence(policy_theta(theta_old), policy_theta(theta))

# Fisher info F = second derivative of KL(theta_old || theta) at theta = theta_old
F = (kl_from_theta(theta_old + eps_fd) - 2 * kl_from_theta(theta_old) + kl_from_theta(theta_old - eps_fd)) / eps_fd**2
print(f"Fisher information F (curvature of KL) at theta={theta_old}: {F:.4f}")

# Suppose the vanilla policy gradient here is g = 0.8. Compute the TRPO step size.
g = 0.8
delta_budget = 0.01   # KL budget
step_size = np.sqrt(2 * delta_budget / (g * F * g)) if F > 0 else 0.0
natural_grad_step = step_size * (g / F if F > 0 else 0)
print(f"Vanilla gradient g = {g}")
print(f"Natural-gradient step  Delta_theta = sqrt(2*delta / (g F g)) * (g/F) = {natural_grad_step:.4f}")
print("This is the LARGEST step in the gradient direction that keeps KL <= delta_budget.")

## 3.4 DDPG — the deterministic policy gradient, via the chain rule

$$\nabla_\theta J \approx \mathbb{E}\Big[\, \nabla_a Q(s,a)\big|_{a=\mu_\theta(s)} \cdot \nabla_\theta \mu_\theta(s) \,\Big]$$

No sampling needed: the actor outputs one exact action, and its gradient flows straight
back through the critic.

In [ ]:
# A toy critic Q(s,a) = -(a - target(s))^2  (peaks exactly at the "correct" action for state s)
# and a linear deterministic actor a = mu_theta(s) = theta * s.
def target_action(s):
    return 0.6 * s

def Q(s, a):
    return -(a - target_action(s)) ** 2

def mu(theta, s):
    return theta * s

s = 2.0
theta = 0.2   # actor currently outputs a = 0.4, but the critic wants a = 1.2

a = mu(theta, s)
dQ_da_analytical = -2 * (a - target_action(s))          # d/da of -(a - t)^2
dmu_dtheta = s                                            # d/dtheta of theta*s
dQ_dtheta_chain_rule = dQ_da_analytical * dmu_dtheta

# Verify with finite differences on the WHOLE composition Q(s, mu_theta(s))
h = 1e-5
dQ_dtheta_fd = (Q(s, mu(theta + h, s)) - Q(s, mu(theta - h, s))) / (2 * h)

print(f"actor's action a = mu_theta(s) = {a:.3f}   (critic wants a = {target_action(s):.3f})")
print(f"dQ/da (analytical)         = {dQ_da_analytical:.4f}")
print(f"dQ/dtheta via chain rule   = {dQ_dtheta_chain_rule:.4f}")
print(f"dQ/dtheta via finite diff  = {dQ_dtheta_fd:.4f}   <- should match")
print("\nGradient ASCENT on theta in this direction pushes a toward the critic's preferred action.")

## 3.5 TD3 — why taking the minimum of two critics curbs overestimation

Any single noisy critic tends to be optimistic on average, because random noise gets
selected preferentially wherever it happens to push the estimate up. Simulate many
independent noisy critic pairs and compare three combination strategies.

In [ ]:
true_Q = 5.0
sigma_noise = 1.0
n_trials = 50_000

Q1 = true_Q + rng.normal(0, sigma_noise, n_trials)
Q2 = true_Q + rng.normal(0, sigma_noise, n_trials)

bias_single   = np.mean(Q1) - true_Q
bias_max      = np.mean(np.maximum(Q1, Q2)) - true_Q
bias_mean     = np.mean((Q1 + Q2) / 2) - true_Q
bias_min_td3  = np.mean(np.minimum(Q1, Q2)) - true_Q

print(f"True Q = {true_Q}")
print(f"Bias of a SINGLE noisy critic:        {bias_single:+.4f}")
print(f"Bias of max(Q1, Q2)  (never do this):  {bias_max:+.4f}   <- systematically optimistic")
print(f"Bias of mean(Q1, Q2):                  {bias_mean:+.4f}")
print(f"Bias of min(Q1, Q2)  (what TD3 uses):  {bias_min_td3:+.4f}   <- pessimistic, but SAFE")
print("\nOverestimation compounds across training (agent exploits its own errors), so TD3")
print("deliberately trades a bit of pessimism for a lot of stability.")

## 3.6 SAC — entropy as an explicit part of the objective

$$H(\pi(\cdot|s)) = -\mathbb{E}_{a\sim\pi}[\log \pi(a|s)]
\qquad\qquad
J(\pi) = \mathbb{E}\Big[\sum_t r_t + \alpha\, H(\pi(\cdot|s_t))\Big]$$

For a Gaussian policy, entropy has a closed form: $H = \tfrac12\log(2\pi e \sigma^2)$ —
wider policies (bigger $\sigma$) get MORE entropy credit, explicitly rewarding exploration.

In [ ]:
def gaussian_entropy(sigma):
    return 0.5 * np.log(2 * np.pi * np.e * sigma ** 2)

sigmas = np.array([0.05, 0.2, 0.5, 1.0, 2.0])
entropies = gaussian_entropy(sigmas)
mean_reward = 3.0   # suppose this is roughly constant regardless of sigma, for illustration

for alpha in [0.0, 0.2, 0.5]:
    J = mean_reward + alpha * entropies
    print(f"alpha={alpha:.1f}:  J(sigma) = {np.round(J, 3)}   (sigma = {sigmas})")

plt.figure(figsize=(6, 3))
for alpha, color in zip([0.0, 0.2, 0.5], ["#8b8a9a", "#e8a765", "#a98fd2"]):
    plt.plot(sigmas, mean_reward + alpha * entropies, "o-", color=color, label=f"alpha={alpha}")
plt.xlabel("policy std sigma"); plt.ylabel("objective J")
plt.title("Larger alpha rewards wider (more exploratory) policies")
plt.legend(); plt.tight_layout(); plt.show()

---
# 4. Model-Based RL

## 4.1 Dyna-Q — real experience + simulated ("imagined") updates

Same Q-learning update rule, applied both to real transitions AND to transitions sampled
from a learned model. Let's compare convergence speed against vanilla Q-learning on the
same corridor from section 2.3.

In [ ]:
def train_dynaq(n_episodes=300, n_planning=10, alpha=0.5, gamma=0.95, seed=0):
    local_rng = np.random.default_rng(seed)
    Q = np.zeros((N_STATES, len(ACTIONS)))
    model = {}   # model[(s, a_idx)] = (s_next, r)   -- learned as we go
    v_history = []
    for ep in range(n_episodes):
        eps = epsilon_schedule(ep)
        s = 0
        for _ in range(50):
            a_idx = epsilon_greedy(Q[s], eps, local_rng)
            s_next, r, done = step(s, ACTIONS[a_idx])

            # --- REAL update ---
            target = r + gamma * np.max(Q[s_next]) * (not done)
            Q[s, a_idx] += alpha * (target - Q[s, a_idx])
            model[(s, a_idx)] = (s_next, r, done)

            # --- SIMULATED (planning) updates, drawn from the learned model ---
            if model:
                keys = list(model.keys())
                for _ in range(n_planning):
                    ps, pa = keys[local_rng.integers(len(keys))]
                    ps_next, pr, pdone = model[(ps, pa)]
                    ptarget = pr + gamma * np.max(Q[ps_next]) * (not pdone)
                    Q[ps, pa] += alpha * (ptarget - Q[ps, pa])

            s = s_next
            if done:
                break
        v_history.append(Q[0].max())
    return Q, v_history

_, v_hist_dynaq = train_dynaq(n_planning=10)
_, v_hist_qlearning = train_qlearning()   # from section 2.3, n_planning=0 equivalent

plt.figure(figsize=(6.5, 3.5))
plt.plot(v_hist_qlearning, color="#8b8a9a", label="vanilla Q-learning")
plt.plot(v_hist_dynaq, color="#5fd0c3", label="Dyna-Q (10 planning steps/episode)")
plt.xlabel("episode"); plt.ylabel("V(start)")
plt.title("Dyna-Q converges in far fewer REAL episodes")
plt.legend(); plt.tight_layout(); plt.show()

## 4.2 Monte Carlo Tree Search — the UCB selection rule

$$\text{UCB}(s,a) = Q(s,a) + c\sqrt{\frac{\ln N(s)}{N(s,a)}}$$

The first term exploits (favors high value); the second explores (favors rarely-visited
children, since $N(s,a)$ sits in the denominator).

In [ ]:
c = 1.4
N_parent = 50
children = [
    {"name": "child A", "N": 10, "Q": 0.60},
    {"name": "child B", "N": 5,  "Q": 0.80},
    {"name": "child C", "N": 30, "Q": 0.50},
    {"name": "child D", "N": 5,  "Q": 0.40},
]

for ch in children:
    ucb = ch["Q"] + c * np.sqrt(np.log(N_parent) / ch["N"])
    ch["UCB"] = ucb
    print(f"{ch['name']}: N={ch['N']:>3}  Q={ch['Q']:.2f}  UCB={ucb:.3f}")

best = max(children, key=lambda c: c["UCB"])
print(f"\nMCTS selects: {best['name']}  (highest UCB -- note it ISN'T just the highest Q!)")

## 4.3 AlphaZero — PUCT: UCB guided by a neural network's prior

$$\text{PUCT}(s,a) = Q(s,a) + c \cdot P(s,a) \cdot \frac{\sqrt{\sum_b N(s,b)}}{1 + N(s,a)}$$

$P(s,a)$ comes from the policy network — it lets the search focus immediately on
promising moves instead of wasting rollouts on random exploration.

In [ ]:
c_puct = 1.5
priors = {"child A": 0.50, "child B": 0.20, "child C": 0.20, "child D": 0.10}
N_sum = sum(ch["N"] for ch in children)

for ch in children:
    P = priors[ch["name"]]
    puct = ch["Q"] + c_puct * P * np.sqrt(N_sum) / (1 + ch["N"])
    ch["PUCT"] = puct
    print(f"{ch['name']}: Q={ch['Q']:.2f}  P={P:.2f}  N={ch['N']:>3}  PUCT={puct:.3f}")

best_puct = max(children, key=lambda c: c["PUCT"])
print(f"\nPUCT selects: {best_puct['name']}")
print("Compare to plain UCB's pick above -- the network's prior can shift the choice")
print("early on (low N), but as N grows the 1/(1+N) term shrinks the prior's influence")
print("and PUCT converges toward pure Q-driven selection, same as plain UCB.")

## 4.4 MuZero — learning representation, dynamics, and prediction end-to-end

$$s_0 = h(o) \qquad s_{t+1}, r_{t+1} = g(s_t, a_t) \qquad p_t, v_t = f(s_t)$$

No access to the real environment's rules — everything happens in a learned latent
space. Let's unroll a toy version of this chain.

In [ ]:
latent_dim = 4
W_h  = rng.normal(size=(2, latent_dim)) * 0.5      # representation: obs(2-dim) -> latent
W_g  = rng.normal(size=(latent_dim, latent_dim)) * 0.3   # dynamics: latent -> next latent
b_a  = {0: rng.normal(size=latent_dim) * 0.2, 1: -rng.normal(size=latent_dim) * 0.2}  # per-action bias
W_v  = rng.normal(size=latent_dim) * 0.4           # prediction: latent -> scalar value

def h(obs):           # representation function
    return np.tanh(obs @ W_h)

def g(s, a):           # dynamics function (predicts next latent state + reward)
    s_next = np.tanh(s @ W_g + b_a[a])
    r_pred = float(np.sum(s_next) * 0.1)   # toy scalar reward head
    return s_next, r_pred

def f(s):               # prediction function (policy + value heads)
    v_pred = float(s @ W_v)
    return v_pred

obs = np.array([0.3, -0.7])
action_sequence = [1, 0, 1]

s = h(obs)
print(f"s0 = h(obs) = {np.round(s, 3)}   (this is now the ONLY thing the model ever sees)")
for t, a in enumerate(action_sequence):
    s, r_pred = g(s, a)
    v_pred = f(s)
    print(f"step {t+1}: action={a} -> s{t+1}={np.round(s, 3)}, "
          f"predicted reward={r_pred:.3f}, predicted value={v_pred:.3f}")

## 4.5 Dreamer — planning by imagining rollouts in latent space

Once a world model exists, roll it forward *without ever touching the real environment*,
score a few candidate action sequences by their imagined return, and act on the best one.

In [ ]:
def imagine_rollout(s0, actions, gamma=0.95):
    s = s0
    total_return = 0.0
    discount = 1.0
    for a in actions:
        s, r_pred = g(s, a)
        total_return += discount * r_pred
        discount *= gamma
    return total_return

s0 = h(np.array([0.1, 0.4]))
candidate_plans = {
    "plan A": [0, 0, 1],
    "plan B": [1, 1, 0],
    "plan C": [1, 0, 0],
}

print("Imagined returns for each candidate plan (NO real environment calls):")
for name, plan in candidate_plans.items():
    ret = imagine_rollout(s0, plan)
    print(f"  {name} {plan}: imagined return = {ret:+.4f}")

best_plan = max(candidate_plans, key=lambda k: imagine_rollout(s0, candidate_plans[k]))
print(f"\nDreamer would act out the first step of: {best_plan}")

---
# 5. Multi-Agent RL

## 5.1 VDN — the simplest possible credit assignment

$$Q_{tot}(s,a) = \sum_i Q_i(s_i, a_i)$$

In [ ]:
Q_agents = np.array([0.4, 0.65, 0.5])
Q_tot = Q_agents.sum()
print("Individual Q-values:", Q_agents)
print("Q_tot (VDN, simple sum):", Q_tot)

# dQ_tot/dQ_i is trivially 1 for every agent -- verify with finite differences anyway.
h = 1e-4
grad = [( (Q_agents + h*np.eye(3)[i]).sum() - (Q_agents - h*np.eye(3)[i]).sum() ) / (2*h) for i in range(3)]
print("dQ_tot/dQ_i (finite difference):", np.round(grad, 4), " -- always exactly 1")

## 5.2 QMIX — monotonic mixing, and why the non-negativity constraint matters

$$Q_{tot} = f_{mix}(Q_1, \dots, Q_N), \qquad \frac{\partial Q_{tot}}{\partial Q_i} \geq 0 \;\; \forall i$$

A tiny 1-hidden-layer mixing network, with weights forced non-negative via `abs()`.

In [ ]:
n_agents, hidden = 3, 4
W1 = rng.normal(size=(n_agents, hidden))
W2 = rng.normal(size=hidden)

def mix_monotonic(Qs, W1, W2):
    # abs() enforces non-negative weights -> guarantees Qtot is monotonic in every Qi
    h_layer = np.maximum(0, Qs @ np.abs(W1))     # ReLU keeps it monotonic too
    return h_layer @ np.abs(W2)

def mix_unconstrained(Qs, W1, W2):
    # SAME weights, but WITHOUT the abs() -- monotonicity is no longer guaranteed
    h_layer = np.maximum(0, Qs @ W1)
    return h_layer @ W2

Q_agents = np.array([0.4, 0.65, 0.5])
Qtot_mono = mix_monotonic(Q_agents, W1, W2)
Qtot_free = mix_unconstrained(Q_agents, W1, W2)
print(f"Qtot (monotonic mixing):   {Qtot_mono:.4f}")
print(f"Qtot (unconstrained mix):  {Qtot_free:.4f}")

def check_monotonic(mix_fn, Qs, W1, W2):
    h = 1e-4
    grads = []
    for i in range(len(Qs)):
        bump = np.eye(len(Qs))[i] * h
        g = (mix_fn(Qs + bump, W1, W2) - mix_fn(Qs - bump, W1, W2)) / (2 * h)
        grads.append(g)
    return np.array(grads)

grad_mono = check_monotonic(mix_monotonic, Q_agents, W1, W2)
grad_free = check_monotonic(mix_unconstrained, Q_agents, W1, W2)
print(f"\ndQtot/dQi with abs() constraint:    {np.round(grad_mono, 4)}   all >= 0 ✓")
print(f"dQtot/dQi WITHOUT the constraint:   {np.round(grad_free, 4)}   "
      f"{'-- some are NEGATIVE! monotonicity broken.' if (grad_free < 0).any() else 'happens to be fine this time, but not guaranteed.'}")

## 5.3 MADDPG — a centralized critic sees everything, at training time only

$$Q_i(s_1,\dots,s_N,\, a_1,\dots,a_N)$$

Each actor only ever sees its OWN observation; the critic (used only during training)
takes the full joint state-action vector.

In [ ]:
n_agents = 3
obs_dim, act_dim = 2, 1

obs = [rng.normal(size=obs_dim) for _ in range(n_agents)]
actions = [rng.normal(size=act_dim) for _ in range(n_agents)]

print("What each DECENTRALIZED actor sees (only its own observation):")
for i in range(n_agents):
    print(f"  actor {i+1} input: {np.round(obs[i], 3)}")

joint_input = np.concatenate(obs + actions)
print(f"\nWhat the CENTRALIZED critic sees (joint state+action, training only):")
print(f"  critic input ({len(joint_input)}-dim): {np.round(joint_input, 3)}")

W_critic = rng.normal(size=len(joint_input)) * 0.3
Q_joint = float(joint_input @ W_critic)
print(f"\nQ_1(s_1,s_2,s_3,a_1,a_2,a_3) = {Q_joint:.4f}")
print("At execution time, this critic is discarded -- each actor keeps acting from its own obs alone.")

---
# 6. Advanced Topics

## 6.1 Distributional RL — same mean, very different risk

$$Q(s,a) = \mathbb{E}[Z(s,a)] \qquad \text{but } Z(s,a) \text{ itself is a full distribution}$$

In [ ]:
atoms = np.array([-2, 0, 2, 4, 6, 8, 10], dtype=float)
p_safe  = np.array([0.02, 0.05, 0.13, 0.60, 0.13, 0.05, 0.02])   # narrow, peaked
p_risky = np.array([0.15, 0.12, 0.10, 0.14, 0.10, 0.14, 0.25])   # wide, spread out

assert np.isclose(p_safe.sum(), 1.0) and np.isclose(p_risky.sum(), 1.0)

mean_safe  = np.sum(atoms * p_safe)
mean_risky = np.sum(atoms * p_risky)
var_safe   = np.sum(p_safe  * (atoms - mean_safe) ** 2)
var_risky  = np.sum(p_risky * (atoms - mean_risky) ** 2)

print(f"SAFE:  mean = {mean_safe:.2f}   variance = {var_safe:.2f}")
print(f"RISKY: mean = {mean_risky:.2f}   variance = {var_risky:.2f}")
print(f"\nBoth have (nearly) the same mean -- a plain Q-learner literally cannot tell them apart.")
print(f"Only by modeling the FULL distribution Z(s,a) does the extra risk become visible.")

plt.figure(figsize=(6.5, 3))
w = 0.35
plt.bar(atoms - w/2, p_safe, width=w, color="#5fd0c3", label="safe")
plt.bar(atoms + w/2, p_risky, width=w, color="#e08a9c", label="risky")
plt.xlabel("return atom z"); plt.ylabel("probability")
plt.title("Distributional RL: same mean, different shape")
plt.legend(); plt.tight_layout(); plt.show()

## 6.2 Curiosity — intrinsic reward as prediction error

$$r_{intrinsic} = \| f_\phi(s_t, a_t) - s_{t+1} \|^2$$

In [ ]:
def forward_model_predict(s, a, W_fwd):
    return np.tanh(np.concatenate([s, [a]]) @ W_fwd)

state_dim = 4
W_fwd = rng.normal(size=(state_dim + 1, state_dim)) * 0.5

s = rng.normal(size=state_dim)
a = 1.0

# A "novel" transition: actual next state is very different from what the model predicts.
s_next_novel = rng.normal(size=state_dim) * 2

# A "familiar" transition: actual next state matches the model's prediction closely.
s_next_familiar = forward_model_predict(s, a, W_fwd) + rng.normal(scale=0.05, size=state_dim)

pred = forward_model_predict(s, a, W_fwd)
r_intrinsic_novel = np.sum((pred - s_next_novel) ** 2)
r_intrinsic_familiar = np.sum((pred - s_next_familiar) ** 2)

print(f"Forward model's prediction: {np.round(pred, 3)}")
print(f"Novel transition:    actual={np.round(s_next_novel, 3)}   intrinsic reward = {r_intrinsic_novel:.4f}")
print(f"Familiar transition: actual={np.round(s_next_familiar, 3)}   intrinsic reward = {r_intrinsic_familiar:.4f}")
print(f"\nNovel/familiar ratio: {r_intrinsic_novel / max(r_intrinsic_familiar, 1e-8):.1f}x more 'curious' about the surprising one.")

## 6.3 RLHF — a reward model plus a KL leash back to the reference policy

$$R(x,y) = R_{RM}(x,y) - \beta \cdot D_{KL}\big(\pi_\theta(\cdot|x) \,\|\, \pi_{ref}(\cdot|x)\big)$$

Without the KL term, the policy can learn to "hack" the reward model by drifting into
regions the reward model was never trained on. The penalty keeps it honest.

In [ ]:
beta = 0.5
R_RM_score = 8.0   # the reward model LOVES this response

pi_ref = np.array([0.4, 0.35, 0.25])     # reference (original) policy's action distribution
pi_close = np.array([0.45, 0.35, 0.20])   # fine-tuned policy: barely moved
pi_far = np.array([0.95, 0.03, 0.02])     # fine-tuned policy: drifted very far (reward hacking!)

kl_close = kl_divergence(pi_close, pi_ref)
kl_far = kl_divergence(pi_far, pi_ref)

R_close = R_RM_score - beta * kl_close
R_far = R_RM_score - beta * kl_far

print(f"Reward model score (same for both): {R_RM_score}")
print(f"\nSmall drift from reference:  KL={kl_close:.4f}  -> final reward = {R_close:.3f}")
print(f"Large drift from reference:  KL={kl_far:.4f}  -> final reward = {R_far:.3f}")
print(f"\nEven though the raw reward-model score is identical, the policy that drifted far")
print(f"gets penalized {R_close - R_far:.2f} points -- the KL leash discourages exploiting")
print(f"blind spots in the reward model.")

---
# 7. Wrap-Up — ask the tutor anything

Every algorithm in this notebook reduces to the same three moves:

1. **Predict** something (a value, a next state, a teammate's contribution)
2. **Act**, and observe what actually happened
3. **Learn from the gap** between prediction and reality (that's every $\delta$, every
   loss function, every advantage estimate you computed above)

Use `ask_groq(...)` below to explore further — swap in your own question.

In [ ]:
print(ask_groq(
    "Tie together why the TD-error, the policy-gradient log-derivative trick, and the "
    "actor-critic architecture are all really the same idea, in 3 short sentences."
))

In [ ]:
# Try your own question:
my_question = "Why does PPO clip the objective instead of just using a smaller learning rate?"
print(ask_groq(my_question))